# Hearo - Fine-tune YAMNet on ESC-50 (+ accuracy benchmark)

Train a custom **on-device** sound classifier for Hearo, export it to TensorFlow.js,
and **benchmark it head-to-head against stock YAMNet** so you can prove the training
actually helped.

**What this does**
1. Downloads [ESC-50](https://github.com/karolpiczak/ESC-50) (2,000 labeled sounds, 50 classes)
2. Uses Google's **YAMNet** as a frozen feature extractor (1024-dim embeddings)
3. Trains a small classifier head (proper **fold-based** split - no data leakage)
4. **Benchmarks fine-tuned vs stock YAMNet** on the same held-out test clips
5. Exports to **TensorFlow.js** - free, offline, on-device

**How to run**
- `Runtime -> Change runtime type -> GPU` (optional, faster)
- `Runtime -> Run all`. That's it - no runtime restart needed.

~15-25 min (mostly the 600 MB dataset download + embedding extraction).


## Setup - install dependencies

Installs TensorFlow Hub + audio libraries on top of Colab's built-in TensorFlow.
We do **not** pin or reinstall TensorFlow (that breaks on current Colab), and the
TF.js converter is installed later at the export step so it can't disturb training.
No runtime restart needed.


In [ ]:
!pip install -q tensorflow_hub librosa==0.10.1 soundfile pandas tqdm scikit-learn

import tensorflow as tf
print('TensorFlow', tf.__version__, '- setup done. Now: Runtime -> Run all.')


## Step 1 - Download ESC-50 (~600 MB)

In [ ]:
import os, urllib.request, zipfile

if not os.path.isdir('ESC-50-master'):
    print('Downloading ESC-50 (~600 MB, one time)...')
    urllib.request.urlretrieve(
        'https://github.com/karolpiczak/ESC-50/archive/master.zip', 'esc50.zip')
    with zipfile.ZipFile('esc50.zip') as z:
        z.extractall('.')
    print('Extracted.')
else:
    print('ESC-50 already downloaded.')

AUDIO_DIR = 'ESC-50-master/audio'
META_CSV  = 'ESC-50-master/meta/esc50.csv'
print('Audio files:', len(os.listdir(AUDIO_DIR)))


## Step 2 - Load YAMNet + its AudioSet class names

YAMNet outputs 1024-dim embeddings (what we train on) AND 521 AudioSet scores
(what the *stock* app uses today - we'll need those names for the benchmark).


In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import csv

print('TensorFlow', tf.__version__)
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

# 521 AudioSet display names (index -> name)
class_map_path = yamnet.class_map_path().numpy().decode('utf-8')
AUDIOSET_NAMES = []
with open(class_map_path) as f:
    reader = csv.reader(f)
    next(reader)  # header: index,mid,display_name
    for row in reader:
        AUDIOSET_NAMES.append(row[2])
print('YAMNet loaded. AudioSet classes:', len(AUDIOSET_NAMES))


## Step 3 - Read ESC-50 labels

In [ ]:
import pandas as pd

meta = pd.read_csv(META_CSV)
class_map = meta[['target', 'category']].drop_duplicates().sort_values('target')
LABELS = class_map['category'].tolist()          # index position == target id
NUM_CLASSES = len(LABELS)
print(NUM_CLASSES, 'classes:')
print(LABELS)


## Step 4 - Extract YAMNet embeddings for every clip

Each 5s clip -> ~10 frame-embeddings of size 1024. We keep track of which clip and
which **fold** each frame came from, so we can split honestly (no frames from the same
clip in both train and test).


In [ ]:
import librosa
from tqdm.auto import tqdm

def load_wav_16k_mono(path):
    wav, _ = librosa.load(path, sr=16000, mono=True)   # YAMNet wants 16 kHz mono float32
    return wav.astype(np.float32)

clip_emb, clip_target, clip_fold, clip_name = [], [], [], []
for _, row in tqdm(meta.iterrows(), total=len(meta), desc='Extracting embeddings'):
    wav = load_wav_16k_mono(os.path.join(AUDIO_DIR, row['filename']))
    _, embeddings, _ = yamnet(wav)
    clip_emb.append(embeddings.numpy())     # [frames, 1024]
    clip_target.append(int(row['target']))
    clip_fold.append(int(row['fold']))
    clip_name.append(row['filename'])

print('Extracted embeddings for', len(clip_emb), 'clips.')


## Step 5 - Fold-based split + train the head

Standard ESC-50 protocol: **folds 1-4 = train, fold 5 = test**. This guarantees no
clip leaks across the split, so the accuracy number is honest.


In [ ]:
TEST_FOLD = 5

def frames_for(folds):
    Xs, ys = [], []
    for emb, tgt, fold in zip(clip_emb, clip_target, clip_fold):
        if (fold == TEST_FOLD) == (folds == 'test'):
            Xs.append(emb)
            ys.append(np.full(emb.shape[0], tgt, dtype=np.int64))
    return np.concatenate(Xs), np.concatenate(ys)

X_tr, y_tr = frames_for('train')
X_val, y_val = frames_for('test')
print('Train frames:', X_tr.shape, '| Test frames:', X_val.shape)

head = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,), name='embedding'),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', name='predictions'),
], name='hearo_head')

head.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
             loss='sparse_categorical_crossentropy', metrics=['accuracy'])
head.summary()

es = tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True,
                                      monitor='val_accuracy')
head.fit(X_tr, y_tr, validation_data=(X_val, y_val),
         epochs=60, batch_size=64, callbacks=[es])


## Step 6 - Clip-level accuracy of the fine-tuned model

Real-world accuracy: average the head's per-frame probabilities over each test clip,
then take the top class. This is how the app will use it.


In [ ]:
from sklearn.metrics import classification_report

ft_preds, trues = [], []
for emb, tgt, fold in zip(clip_emb, clip_target, clip_fold):
    if fold != TEST_FOLD:
        continue
    clip_prob = head.predict(emb, verbose=0).mean(axis=0)   # [50]
    ft_preds.append(int(clip_prob.argmax()))
    trues.append(tgt)

ft_preds, trues = np.array(ft_preds), np.array(trues)
ft_acc = (ft_preds == trues).mean()
print(f'Fine-tuned clip-level accuracy on fold {TEST_FOLD}: {ft_acc:.1%}\n')
print(classification_report(trues, ft_preds, target_names=LABELS, zero_division=0))


## Step 7 - BENCHMARK: fine-tuned vs stock YAMNet

This is the "is it actually better?" test. For the **alert-relevant** sounds (the ones
Hearo cares about), we map BOTH models to Hearo alert categories and measure how often
each gets it right - on the exact same held-out test clips.

- **Stock YAMNet** = top of 521 AudioSet classes -> Hearo category (what the app does *today*)
- **Fine-tuned** = top of your 50 ESC-50 classes -> Hearo category


In [ ]:
# ESC-50 class -> Hearo alert category (true labels + fine-tuned mapping)
esc50_to_hearo = {
    'crying_baby':    'baby_cry',
    'dog':            'dog_bark',
    'glass_breaking': 'glass_break',
    'siren':          'siren',
    'car_horn':       'car_horn',
    'clock_alarm':    'alarm',
    'door_wood_knock':'knock',
    'church_bells':   'doorbell',
}

# Stock YAMNet AudioSet-label -> Hearo category (mirrors AUDIOSET_CLASS_PATTERNS in App.jsx)
AUDIOSET_PATTERNS = [
    (['fire alarm','smoke detector','smoke alarm','civil defense siren','fire truck siren'], 'fire_alarm'),
    (['doorbell','ding-dong','bell'], 'doorbell'),
    (['telephone bell ringing','telephone','ringtone','phone'], 'phone_ring'),
    (['baby cry','infant cry','whimper','crying'], 'baby_cry'),
    (['car horn','honking','car alarm','vehicle horn','beep, bleep'], 'car_horn'),
    (['breaking','glass','shatter','smash','crash'], 'glass_break'),
    (['screaming','shouting','yelling','scream','shriek'], 'scream'),
    (['dog','bark','bow-wow','growling'], 'dog_bark'),
    (['knock','tap','rapping'], 'knock'),
    (['siren','ambulance','police siren'], 'siren'),
    (['alarm clock','buzzer','alarm','beeping'], 'alarm'),
]
def match_audioset(label):
    low = label.lower()
    for pats, cat in AUDIOSET_PATTERNS:
        if any(p in low for p in pats):
            return cat
    return None

from collections import defaultdict
totals   = defaultdict(int)
stock_ok = defaultdict(int)
ft_ok    = defaultdict(int)

for emb, tgt, fold, name in tqdm(list(zip(clip_emb, clip_target, clip_fold, clip_name)),
                                 desc='Benchmarking'):
    if fold != TEST_FOLD:
        continue
    true_cat = esc50_to_hearo.get(LABELS[tgt])
    if true_cat is None:        # not an alert sound - skip
        continue
    totals[true_cat] += 1

    # fine-tuned prediction
    ft_top = int(head.predict(emb, verbose=0).mean(axis=0).argmax())
    if esc50_to_hearo.get(LABELS[ft_top]) == true_cat:
        ft_ok[true_cat] += 1

    # stock YAMNet prediction (re-run to get the 521 scores)
    wav = load_wav_16k_mono(os.path.join(AUDIO_DIR, name))
    scores, _, _ = yamnet(wav)
    stock_top = int(scores.numpy().mean(axis=0).argmax())
    if match_audioset(AUDIOSET_NAMES[stock_top]) == true_cat:
        stock_ok[true_cat] += 1

print(f'\n{"Alert category":<16}{"clips":>7}{"stock YAMNet":>15}{"fine-tuned":>13}')
print('-' * 51)
T = S = F = 0
for cat in sorted(totals):
    n = totals[cat]; s = stock_ok[cat]; f = ft_ok[cat]
    T += n; S += s; F += f
    print(f'{cat:<16}{n:>7}{s/n:>14.0%}{f/n:>13.0%}')
print('-' * 51)
print(f'{"OVERALL":<16}{T:>7}{S/T:>14.0%}{F/T:>13.0%}')
print(f'\nFine-tuned is {"BETTER" if F>S else "not better"} on Hearo alert sounds.')


## Step 8 - Export to TensorFlow.js

We export the trained head as a **SavedModel**, then convert *that* to a TF.js
**graph model**. Going through SavedModel avoids the Keras-3 / converter incompatibility,
and a graph model is exactly how the app already loads YAMNet (`tf.loadGraphModel`).


In [ ]:
import os

# 1) Export the trained head as a TF SavedModel
SM_DIR = 'hearo_head_savedmodel'
head.export(SM_DIR)
print('SavedModel written to', SM_DIR)

# 2) Install the converter HERE (after training) so it can't disturb the runtime,
#    then convert SavedModel -> TF.js graph model.
!pip install -q tensorflowjs
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model {SM_DIR} hearo_sound_model

print('TF.js model:', os.listdir('hearo_sound_model'))


## Step 9 - Save labels + Hearo category mapping

In [ ]:
import json

with open(os.path.join(OUT_DIR, 'labels.json'), 'w') as f:
    json.dump(LABELS, f, indent=2)
with open(os.path.join(OUT_DIR, 'esc50_to_hearo.json'), 'w') as f:
    json.dump(esc50_to_hearo, f, indent=2)
# stash the benchmark numbers so you remember them later
with open(os.path.join(OUT_DIR, 'benchmark.json'), 'w') as f:
    json.dump({'fine_tuned_clip_acc': float(ft_acc),
               'hearo_stock_acc': S / T, 'hearo_finetuned_acc': F / T}, f, indent=2)
print('Saved labels.json, esc50_to_hearo.json, benchmark.json')


## Step 10 - Download the bundle

In [ ]:
import shutil
shutil.make_archive('hearo_sound_model', 'zip', OUT_DIR)
from google.colab import files
files.download('hearo_sound_model.zip')
print('Downloaded hearo_sound_model.zip')


## Step 11 - Integrate into the Hearo app

1. Unzip into the repo at **`public/models/hearo/`**:
   ```
   public/models/hearo/
     model.json
     group1-shard1of1.bin
     labels.json
     esc50_to_hearo.json
   ```
2. Tell me in chat that the model is in place. I'll wire `App.jsx` to:
   - load this head model alongside YAMNet,
   - feed YAMNet **embeddings** (output index 1) into the head,
   - map results via `esc50_to_hearo.json`,
   - add a **Settings toggle** to switch between stock YAMNet and your custom model
     so you can A/B them live, in the app, on real sounds.
